In [ ]:
import os
from pathlib import Path
import pandas as pd
import psycopg
from dotenv import load_dotenv
from linearmodels.panel import PanelOLS

load_dotenv()

conn = psycopg.connect(
    f"host=localhost port=5433 dbname=youbike user=postgres "
    f"password={os.getenv('EC2_DB_PASSWORD')}"
)

In [ ]:
START, END = '2026-08-20', '2026-08-29'
CACHE = Path(f'data/analysis_{START.replace("-","")}_{END.replace("-","")}.parquet')

if CACHE.exists():
    df_full = pd.read_parquet(CACHE)
else:
    sql = open('sql/analysis_dataset.sql').read()
    df_full = pd.read_sql(sql, conn, params={'start': START, 'end': END})
    df_full.to_parquet(CACHE)

print(df_full.shape)

In [ ]:
panel_full = prepare(df_full)

r = panel_full.loc[panel_full['precipitation_10min'] > 0, 'precipitation_10min']
print(f'非零觀測 {len(r)} 筆 ({len(r)/len(panel_full):.2%})')
print(r.quantile([.5, .75, .9, .95, .99]))
print()
print(panel_full.assign(d=panel_full['info_time'].dt.tz_convert('Asia/Taipei').dt.date)
      .groupby('d')['rain'].agg(['size', 'sum', 'mean']))

In [ ]:
def prepare(raw):
    """從原始查詢結果產生分析樣本。不修改輸入。"""
    x = raw.copy()
    tp = x['info_time'].dt.tz_convert('Asia/Taipei')
    x['hour'] = tp.dt.hour
    x['dow']  = tp.dt.dayofweek
    x['date'] = tp.dt.date
    x['rain'] = (x['precipitation_10min'] > 0).astype(int)

    # 樣本限制：排除深夜（調度主導，見 8/28 日內模式檢查）
    keep = x['hour'].between(6, 22) & x['delta'].notna()
    return x[keep].copy()

panel = prepare(df_raw)
print(panel.shape, '降雨觀測比例:', round(panel['rain'].mean(), 4))

In [ ]:
FORMULA_BASE = ' + air_temperature + gap_min + EntityEffects + TimeEffects'

def fit_panel(data, treat, formula_extra=FORMULA_BASE):
    """雙向固定效應 + 以 rain_station 為層級的 cluster 標準誤。

    索引用 (sno, obs_slot)：obs_slot 是 10 分鐘對齊的時間，
    若用 info_time 會產生數十萬個時間層級而爆記憶體。
    """
    d = data.set_index(['sno', 'obs_slot'])
    mod = PanelOLS.from_formula(
        f'delta ~ {treat}{formula_extra}',
        data=d, drop_absorbed=True,
    )
    return mod.fit(cov_type='clustered', clusters=d['rain_station'])

In [ ]:
res_full = fit_panel(panel_full, 'precipitation_10min')
print(res_full.summary.tables[1])

In [ ]:
panel_full['rain_cat'] = pd.cut(
    panel_full['precipitation_10min'],
    bins=[-0.01, 0.001, 1.0, 100],
    labels=['none', 'light', 'heavy'],
)
res_cat = fit_panel(panel_full, 'C(rain_cat, Treatment(reference="none"))')
print(res_cat.summary.tables[1])

In [ ]:
panel_full['rain_light'] = (panel_full['rain_cat'] == 'light').astype(int)
panel_full['rain_heavy'] = (panel_full['rain_cat'] == 'heavy').astype(int)

res_cat = fit_panel(panel_full, 'rain_light + rain_heavy')
print(res_cat.summary.tables[1])

In [ ]:
h = panel_full.loc[panel_full['precipitation_10min'] > 1.0, 'precipitation_10min']
print(len(h))
print(h.quantile([.25, .5, .75, .9]))

In [ ]:
panel_full['r_light'] = panel_full['precipitation_10min'].between(0.001, 1.0).astype(int)
panel_full['r_mid']   = panel_full['precipitation_10min'].between(1.0001, 3.0).astype(int)
panel_full['r_high']  = (panel_full['precipitation_10min'] > 3.0).astype(int)

print(panel_full[['r_light','r_mid','r_high']].sum())

res3 = fit_panel(panel_full, 'r_light + r_mid + r_high')
print(res3.summary.tables[1])

In [ ]:
res_binary = fit_panel(panel, 'rain')
print(res_binary.summary.tables[1])

In [ ]:
res_dose = fit_panel(panel, 'precipitation_10min')
print(res_dose)

In [ ]:
# 對「處理變數」設上限，不是對應變數
# 對應變數 trimming 會有內生性問題（砍掉受處理影響最強的觀測）
for thr in [20, 10, 5, 2]:
    sub = panel[panel['precipitation_10min'] < thr]
    r = fit_panel(sub, 'precipitation_10min')
    print(f"雨量上限={thr:3} n={r.nobs:7} "
          f"beta={r.params['precipitation_10min']:8.4f} "
          f"se={r.std_errors['precipitation_10min']:.4f} "
          f"p={r.pvalues['precipitation_10min']:.4f}")

In [ ]:
big = panel[panel['precipitation_10min'] >= 5]
print(f'大雨觀測 {len(big)} 筆 / 全樣本 {len(panel)} 筆')
print(big.groupby('date').size())
print(f"涵蓋 {big['rain_station'].nunique()} 個測站、"
      f"{big['sno'].nunique()} 個 YouBike 站")

In [ ]:
r = panel.loc[panel['precipitation_10min'] > 0, 'precipitation_10min']
print(f'非零觀測 {len(r)} 筆 ({len(r)/len(panel):.2%})')
print(r.describe())
print(r.quantile([.5, .75, .9, .95, .99]))